In [9]:
import requests
import pandas as pd
import time
from tqdm import tqdm #import para la barra

session = requests.Session() #Abre una sesión para consultar a la API en lugar de abrir y cerrar sesion con cada consulta.

BASE_URL = "https://api.deezer.com"
MAX_SONGS = 50  #límite de canciones por artista
SLEEP_TIME = 0.3  #pausa entre consultas para no sobrecargar la API

ARTISTS = [
    "La Fuga",
    "Héroes del Silencio",
    "Billie Eilish",
    "Love of Lesbian",
    "Estopa",
    "Mägo de Oz",
    "Mr. Kilombo",
    "Rozalén",
    "Taburete",
    "Extremoduro",
    "La Plazuela",
    "Veintiuno",
    "Ojete Calor",
    "Rata Blanca",
    "Vetusta Morla",
    "Leiva",
    "Bad Bunny",
    "Enrique Bunbury",
    "Marea",
    "Joaquín Sabina",
    "Rosalía",
    "Queen",
    "The Lumineers",
    "Foo Fighters",
    "Muse",
    "Metallica",
    "Ginebras",
    "IZAL",
    "Kaiser Chiefs",
    "Residente"
]

def search_tracks(album_id):
    url = f"{BASE_URL}/album/{album_id}/tracks"
    r = session.get(url)
    r.raise_for_status() #evalua el fallo de forma inmediata y clara, si el codigo no es 200 nos salta fallo claro
    data = r.json()
    if not data["data"]: #pese a estar comprobados, si no se encuentran datos, aplicamos este if
        print(f"⚠️ No tracks found for album {album_id}")
        return []
    return data["data"]

def album_data(album_id):
    url = f"{BASE_URL}/album/{album_id}"
    r = session.get(url)
    r.raise_for_status()
    data = r.json()
    return {
        "album_title": data["title"],
        "year": data.get("release_date", None)[:4] if data.get("release_date") else None, #None para que no falle el codigo si no está y [:4] para devolver solo los primeros 4 caracteres (año)
        "genre": data.get("genres", {}).get("data", [{}])[0].get("name", None), #igual que arriba, nos devuelve error en lugar de fallar el codigo
        "genre_id": data.get("genre_id", None)
    }

def search_artist(name):
    url = f"{BASE_URL}/search/artist" #define url con una base común para todas incluida en variable BASE_URL y un añadido para esta función
    params = {"q": name, "limit": 1} #diccionario para definir la busqueda, el nombre lo buscaremos desde su clave q(definida por deezer) limite 1 para que solo exporte 1 artista
    r = session.get(url, params=params) #llamar a la url con los parámetros facilitados
    r.raise_for_status()
    data = r.json()
    if not data["data"]:
        print(f"⚠️ Artist not found: {name}")
        return None
    return data["data"][0]

def search_albums(artist_id):
    url = f"{BASE_URL}/artist/{artist_id}/albums" #con la url base y el id obtenido en la función search_artist
    r = session.get(url)
    r.raise_for_status()
    data = r.json()
    if not data["data"]:
        print(f"⚠️ No albums found for artist {artist_id}")
        return []
    return data["data"]

In [ ]:
results = []
bar = tqdm(ARTISTS, desc="Starting...")

for artist in bar:
    bar.set_description(f"Processing: {artist}")
    try:
        artist_info = search_artist(artist) #buscamos al artista llamando a la función y creamos variable
        if artist_info is None: #si no se encontró, salta al siguiente
            continue
        artist_id = artist_info["id"] #extraemos su id con la variable anterior e indicando la clave que buscamos al ser diccionario
        albums = search_albums(artist_id) #llamamos a la funcion search_albums con el id de artista obtenido en la anterior y creamos variable

        total_songs = 0
        for album in albums:
            if total_songs >= MAX_SONGS:
                break
            album_id = album["id"]
            tracks = search_tracks(album_id)
            if not tracks:
                continue
            total_songs += len(tracks)
            if total_songs > MAX_SONGS:
                tracks = tracks[:MAX_SONGS - (total_songs - len(tracks))]
            info_album = album_data(album_id)
            time.sleep(SLEEP_TIME)

            for track in tracks:
                results.append({
                    "artist_id": artist_info["id"],
                    "artist_name": artist_info["name"],
                    "album_title": info_album["album_title"],
                    "track_title": track["title"],
                    "type": track["type"],
                    "year": info_album["year"],
                    "genre": info_album["genre"],
                    "genre_id": info_album["genre_id"]
                })
    except requests.exceptions.RequestException as e:
        print(f"⚠️ Network error processing {artist}: {e}")
        continue

df = pd.DataFrame(results) #pasar a tabla

df.to_csv("deezer_artists.csv", index=False, encoding="utf-8-sig")
#Convierte la tabla a CSV, guárdala con ese nombre, sin la columna de índices y con soporte para caracteres especiales del español
print(f"✅ Exported successfully! {len(df)} tracks from {df['artist_name'].nunique()} artists")
session.close() #cerramos la sesión una vez terminado todo
df.head()

Processing: Residente: 100%|██████████| 30/30 [01:50<00:00,  3.68s/it]         

✅ Exported successfully! 1500 tracks from 30 artists
